# Domain 5: Context Management & Reliability
## TS 5.1: Manage Conversation Context to Preserve Critical Information Across Long Interactions
**Certification Focus**: CCAR-F Domain 5 (15% Exam Weight) | **Scenario Alignment**: Scenario 1 & 4

---

### 🎯 Key Architectural Principles
1. **Tool Output Trimming**: Bulky database payloads and carrier tracking logs quickly consume context windows. Always trim non-essential internal fields before feeding results back into Claude.
2. **Persistent Case Facts**: Critical business entities (Customer ID, verified identity, order reference) must be maintained in an immutable `CaseFacts` block at the top of the prompt, immune to conversation summarization loss.
3. **Preventing Hallucination via Pruning**: Long conversation histories dilute attention; extracting structured facts into persistent state allows pruning old conversational turns without losing critical context.

In [ ]:
# Step 1: Environment & Setup
import json
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 5.1 Context Preservation.")

### ✂️ Step 2: Output Trimmer & Case Facts Manager
We implement the tool output trimmer and the persistent `CaseFacts` state container.

In [ ]:
# Step 2: Trimming and Persistent Facts Logic
RAW_ORDER_PAYLOAD = {
    "order_id": "ORD-2026-9921",
    "customer_id": "CUST-441",
    "status": "delivered",
    "amount": 149.99,
    "internal_db_primary_key": 998124,
    "payment_gateway_auth_token": "tok_live_secret_auth_99182371",
    "carrier_tracking_code": "TRACK_FEDEX_9981247182",
    "warehouse_rack_id": "RACK-4B-SHELF-2",
    "server_processing_node": "us-east-1a-worker-04",
}

def trim_order_output(raw_order: Dict[str, Any]) -> Dict[str, Any]:
    """Strips bulky internal telemetry, leaving only essential business fields."""
    return {
        "order_id": raw_order.get("order_id"),
        "customer_id": raw_order.get("customer_id"),
        "status": raw_order.get("status"),
        "amount": raw_order.get("amount"),
    }


@dataclass
class CaseFacts:
    customer_id: str
    customer_email: str
    order_id: Optional[str] = None
    order_amount: Optional[float] = None
    status: str = "Active"

    def render_prompt_block(self) -> str:
        """Renders persistent state block placed at the top of prompt context."""
        lines = [
            "### PERSISTENT CASE FACTS (IMMUTABLE STATE)",
            f"- Customer ID: {self.customer_id}",
            f"- Customer Email: {self.customer_email}",
        ]
        if self.order_id:
            lines.append(f"- Active Order: {self.order_id} (${self.order_amount:.2f})")
        lines.append(f"- Status: {self.status}")
        return "\n".join(lines)


def lookup_order_tool(order_id: str, case_facts: CaseFacts) -> Dict[str, Any]:
    """Simulates tool execution that updates persistent CaseFacts state."""
    trimmed = trim_order_output(RAW_ORDER_PAYLOAD)
    case_facts.order_id = trimmed["order_id"]
    case_facts.order_amount = trimmed["amount"]
    return trimmed

print(" Output trimmer and CaseFacts container ready.")

### 🧪 Step 3: Interactive Simulation
1. Compare raw payload size vs trimmed payload size.
2. Observe how tool execution updates the persistent Case Facts block.

In [ ]:
# Step 3: Simulation
# 1. Trimming comparison
trimmed_order = trim_order_output(RAW_ORDER_PAYLOAD)
raw_size = len(json.dumps(RAW_ORDER_PAYLOAD))
trimmed_size = len(json.dumps(trimmed_order))

print(f"Raw Payload Characters: {raw_size} chars")
print(f"Trimmed Payload Characters: {trimmed_size} chars")
print(f"Token Footprint Reduction: ~{((raw_size - trimmed_size) / raw_size) * 100:.1f}%")

# 2. CaseFacts persistence across turns
facts = CaseFacts(customer_id="CUST-441", customer_email="alice@test.com")
print("\nInitial Prompt Block:\n", facts.render_prompt_block())

# Tool execution updates facts
lookup_order_tool("ORD-2026-9921", facts)
print("\nUpdated Prompt Block after Order Lookup:\n", facts.render_prompt_block())

### 🔬 Step 4: Verification Assertions
We assert that:
1. Sensitive internal keys (rack ID, gateway token) are stripped from trimmed output.
2. CaseFacts reflects customer and order state reliably.

In [ ]:
# Step 4: Verification Checks
assert "warehouse_rack_id" not in trimmed_order
assert "payment_gateway_auth_token" not in trimmed_order
assert trimmed_order["order_id"] == "ORD-2026-9921"
assert trimmed_order["amount"] == 149.99

rendered = facts.render_prompt_block()
assert "CUST-441" in rendered
assert "ORD-2026-9921" in rendered
assert "$149.99" in rendered

print(" All verification assertions passed! Context preservation verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Trim tool outputs to essential fields before appending them to conversation history.
- **Rule 2**: Maintain critical identifiers in a persistent **CaseFacts** block outside conversation summaries.
- **Rule 3**: Strip raw backend telemetry and internal keys to conserve context window capacity.